<a href="https://colab.research.google.com/github/Muaz-Ibn-Kamal/circadian-ici-response-validation/blob/main/muaz_sti_paper_EXECUTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, re
import numpy as np
import pandas as pd
from scipy.stats import pointbiserialr
from sklearn.model_selection import StratifiedKFold, LeaveOneOut, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CLOCK_GENE_COLS = [
    "ARNTL", "CLOCK", "NPAS2", "PER1", "PER2", "PER3",
    "CRY1", "CRY2", "NR1D1", "NR1D2", "RORA", "RORB", "RORC",
    "TIMELESS", "TIPIN", "CSNK1D", "CSNK1E",
]
GEP_GENES = [
    "CCL5", "CD27", "CD274", "CD276", "CD8A", "CMKLR1", "CXCL9", "CXCR6",
    "HLA-DQA1", "HLA-DRB1", "HLA-E", "IDO1", "LAG3", "NKG7", "PDCD1LG2",
    "PSMB10", "STAT1", "TIGIT",
]
IPRES_GENES = [
    "IL10", "VEGFA", "VEGFC", "FLT1", "ANGPT2",
    "CCL2", "CCL7", "CCL8", "CCL13",
    "AXL", "ROR2", "WNT5A", "LOXL2", "TWIST2", "TAGLN", "FAP",
]
IPRES_INVERSE_GENES = ["CDH1"]
TIDE_CTL_GENES = ["CD8A", "CD8B", "GZMA", "GZMB", "PRF1"]
TIDE_CAF_GENES = ["FAP", "PDGFRB", "COL1A1", "COL1A2", "ACTA2"]
TIDE_MDSC_GENES = ["CD33", "ITGAM", "CD14", "ARG1"]
TIDE_M2_GENES = ["CD163", "MRC1", "MSR1", "IL10"]
BASAL_GENES = ["KRT5", "KRT6A", "KRT14"]
LUMINAL_GENES = ["FOXA1", "GATA3", "KRT20", "PPARG"]

C_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]

DATA_DIR = "/home/claude/nbrun"
if not os.path.isdir(DATA_DIR):
    DATA_DIR = "/home/claude/nbrun"

def p(fname):
    return os.path.join(DATA_DIR, fname)


## 1. Load & Build Labeled Tables

In [ ]:
def cpm_log_normalize(counts_df):
    lib_sizes = counts_df.sum(axis=0)
    cpm = counts_df.div(lib_sizes, axis=1) * 1e6
    return np.log2(cpm + 1)

def composite_score(X):
    z = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
    return z.mean(axis=1).reshape(-1, 1)

raw_expr = pd.read_csv(p("imvigor210_expression.csv"), index_col=0)
log_cpm = cpm_log_normalize(raw_expr)

clinical = pd.read_csv(p("imvigor210_clinical.csv"), index_col=0).set_index("pat_id")
clinical["label"] = clinical["BOR"].map({"R": 1, "NR": 0})
clinical = clinical.dropna(subset=["label"])

clock_expr = log_cpm.loc[CLOCK_GENE_COLS].T
common = clock_expr.index.intersection(clinical.index)
imvigor = clock_expr.loc[common].join(clinical.loc[common, ["BOR", "TMB", "label"]])
imvigor["label"] = imvigor["label"].astype(int)

print(f"IMvigor210: {imvigor.shape[0]} patients, {imvigor['label'].sum()} responders "
      f"({imvigor['label'].mean()*100:.1f}%), TMB missing: {imvigor['TMB'].isna().sum()}")
imvigor.head()


FileNotFoundError: [Errno 2] No such file or directory: '/home/claude/nbrun/imvigor210_expression.csv'

In [ ]:
def parse_series_matrix(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    def split_quoted_tsv(line):
        return [x.strip('"') for x in line.strip().split("\t")[1:]]

    titles = split_quoted_tsv(next(l for l in lines if l.startswith("!Sample_title")))
    char_rows = [split_quoted_tsv(l) for l in lines if l.startswith("!Sample_characteristics_ch1")]

    records = []
    for i, pid in enumerate(titles):
        d = {"patient_id": pid}
        for row in char_rows:
            if i < len(row) and ":" in row[i]:
                k, v = row[i].split(":", 1)
                d[k.strip()] = v.strip()
        records.append(d)
    df = pd.DataFrame(records).rename(columns={"anti-pd-1 response": "response_raw",
                                                 "biopsy time": "biopsy_time"})
    if "biopsy_time" not in df.columns:
        df["biopsy_time"] = "pre-treatment"
    return df[["patient_id", "response_raw", "biopsy_time"]]

RESPONDER = {"Complete Response", "Partial Response"}
NONRESPONDER = {"Progressive Disease"}

label_df = parse_series_matrix(p("GSE78220_series_matrix.txt"))
label_df["label"] = label_df["response_raw"].map(
    lambda r: 1 if r in RESPONDER else (0 if r in NONRESPONDER else None))
label_df = label_df.dropna(subset=["label"])

fpkm = pd.read_excel(p("GSE78220_PatientFPKM.xlsx"), sheet_name="FPKM").set_index("Gene")
clock_gse = fpkm.loc[[g for g in CLOCK_GENE_COLS if g in fpkm.index]].T

on_tx_ids = set(label_df.loc[label_df.biopsy_time.str.lower() == "on-treatment", "patient_id"])
bare = {r: re.sub(r"\.(baseline|OnTx)$", "", r) for r in clock_gse.index}
kept = [r for r in clock_gse.index if bare[r] not in on_tx_ids]

stripped = {r: bare[r] for r in kept}
candidates = set(stripped.values())
ab_bases = {c[:-1] for c in candidates if c.endswith(("A", "B"))}
dup_bases = {b for b in ab_bases if (b + "A") in candidates and (b + "B") in candidates}

groups = {}
for r in kept:
    b = stripped[r]
    base = b[:-1] if (b.endswith(("A", "B")) and b[:-1] in dup_bases) else b
    groups.setdefault(base, []).append(r)
clock_gse_clean = pd.DataFrame({b: clock_gse.loc[rows].mean(axis=0) for b, rows in groups.items()}).T

label_indexed = label_df.drop_duplicates("patient_id").set_index("patient_id")
common_gse = clock_gse_clean.index.intersection(label_indexed.index)
gse78220 = clock_gse_clean.loc[common_gse].join(label_indexed.loc[common_gse, ["label"]])
gse78220["label"] = gse78220["label"].astype(int)

print(f"GSE78220: {gse78220.shape[0]} patients, {gse78220['label'].sum()} responders "
      f"({gse78220['label'].mean()*100:.1f}%)")
gse78220.head()


In [ ]:
import gzip

def find_file(candidates):
    for c in candidates:
        full = p(c)
        if os.path.isfile(full):
            return full
    return None

def open_maybe_gz(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", errors="replace")
    return open(path, "r", encoding="utf-8", errors="replace")

def parse_gse176307_series_matrix(path):
    with open_maybe_gz(path) as f:
        lines = f.readlines()

    def split_quoted_tsv(line):
        return [x.strip('"') for x in line.strip().split("\t")[1:]]

    titles = split_quoted_tsv(next(l for l in lines if l.startswith("!Sample_title")))
    char_rows = [split_quoted_tsv(l) for l in lines if l.startswith("!Sample_characteristics_ch1")]

    records = []
    for i, title in enumerate(titles):
        d = {"patient_id": title.replace("Patient sample ", "").strip()}
        for row in char_rows:
            if i < len(row) and ":" in row[i]:
                k, v = row[i].split(":", 1)
                d[k.strip()] = v.strip()
        records.append(d)
    return pd.DataFrame(records)

gse176307_expr_path = find_file([
    "GSE176307_BACI_log_trans_normalized_RNAseq.csv",
    "GSE176307_expression.csv.gz", "GSE176307_expression.csv",
    "GSE176307_BACI_log_trans_normalized_RNAseq.csv.gz",
    "GSE176307_BACI_log_trans_normalized_RNAseq_csv.gz",
])
gse176307_matrix_path = find_file([
    "GSE176307_series_matrix.txt",
    "GSE176307_series_matrix.txt.gz",
    "GSE176307_series_matrix_txt.gz",
])

gse176307 = None
if gse176307_expr_path and gse176307_matrix_path:
    g176_clin = parse_gse176307_series_matrix(gse176307_matrix_path)
    RESPONDER_176 = {"CR", "PR"}
    NONRESPONDER_176 = {"PD"}
    g176_clin["label"] = g176_clin["io.response"].map(
        lambda r: 1 if r in RESPONDER_176 else (0 if r in NONRESPONDER_176 else None))
    g176_clin["TMB"] = pd.to_numeric(g176_clin.get("tmb"), errors="coerce")
    g176_clin = g176_clin.dropna(subset=["label"]).set_index("patient_id")

    g176_expr = pd.read_csv(gse176307_expr_path, index_col=0)
    common_176 = g176_expr.index.intersection(g176_clin.index)

    clock_present_176 = [g for g in CLOCK_GENE_COLS if g in g176_expr.columns]
    gse176307 = g176_expr.loc[common_176, clock_present_176].join(
        g176_clin.loc[common_176, ["label", "TMB"]])
    gse176307["label"] = gse176307["label"].astype(int)

    print(f"GSE176307: {gse176307.shape[0]} patients, {gse176307['label'].sum()} responders "
          f"({gse176307['label'].mean()*100:.1f}%), TMB missing: {gse176307['TMB'].isna().sum()}")
else:
    missing = []
    if not gse176307_expr_path:
        missing.append("GSE176307_BACI_log_trans_normalized_RNAseq.csv")
    if not gse176307_matrix_path:
        missing.append("GSE176307_series_matrix.txt")
    print(f"GSE176307 not loaded -- missing file(s) in {DATA_DIR}: {missing}. "
          f"Pooled/external-validation cells below will fall back to GSE78220 only.")

gse176307.head() if gse176307 is not None else None


## 2. Baseline: Circadian Signature Alone

In [ ]:
def run_kfold_cv(X, y, n_folds, seed=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    y_score = np.zeros_like(y, dtype=float)
    for tr, te in skf.split(X, y):
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(C=1.0, max_iter=2000, random_state=seed, class_weight="balanced")
        clf.fit(sc.transform(X[tr]), y[tr])
        y_score[te] = clf.predict_proba(sc.transform(X[te]))[:, 1]
    return y_score

def bootstrap_auc_ci(y_true, y_score, n_bootstrap=2000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    aucs = np.array(aucs)
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    return aucs.mean(), lo, hi, (aucs <= 0.5).mean()

df_m = imvigor.dropna(subset=["TMB"]).copy()
y = df_m["label"].values.astype(int)
circadian_X = composite_score(df_m[CLOCK_GENE_COLS].values)
tmb_X = df_m[["TMB"]].values.astype(float)

circ_score = run_kfold_cv(circadian_X, y, 5)
tmb_score = run_kfold_cv(tmb_X, y, 5)

for name, sc in [("Circadian_only", circ_score), ("TMB_only", tmb_score)]:
    auc = roc_auc_score(y, sc); auprc = average_precision_score(y, sc)
    m, lo, hi, pv = bootstrap_auc_ci(y, sc)
    print(f"{name:16s} AUC={auc:.3f} AUPRC={auprc:.3f}  95% CI=[{lo:.3f},{hi:.3f}]  p={pv:.4f}")


## 3. Composite Score Methods: Z-score vs Rank vs PCA (nested per fold)

In [ ]:
def rank_composite_fit(X_train):
    n_tr = X_train.shape[0]
    def transform(X):
        out = np.zeros_like(X, dtype=float)
        for j in range(X.shape[1]):
            ranks_tr = pd.Series(X_train[:, j]).rank(method="average").values
            order = np.argsort(X_train[:, j])
            sorted_train = X_train[order, j]
            sorted_ranks = ranks_tr[order]
            out[:, j] = np.interp(X[:, j], sorted_train, sorted_ranks) / n_tr
        return out.mean(axis=1, keepdims=True)
    return transform

def pca_composite_fit(X_train, sign_ref=None):
    mu, sd = X_train.mean(0), X_train.std(0) + 1e-8
    pca = PCA(n_components=1, random_state=RANDOM_SEED)
    pca.fit((X_train - mu) / sd)
    sign = 1.0
    if sign_ref is not None:
        pc1_tr = pca.transform((X_train - mu) / sd)[:, 0]
        if np.corrcoef(pc1_tr, sign_ref)[0, 1] < 0:
            sign = -1.0
    def transform(X):
        return sign * pca.transform((X - mu) / sd)[:, [0]]
    return transform

def zscore_composite_fit(X_train):
    mu, sd = X_train.mean(0), X_train.std(0) + 1e-8
    def transform(X):
        return ((X - mu) / sd).mean(axis=1, keepdims=True)
    return transform

def nested_cv_pipeline(build_features_fn, y, n_folds=5, seed=RANDOM_SEED,
                        c_grid=None, inner_folds=3):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    y_score = np.zeros(len(y), dtype=float)
    chosen_Cs = []
    for tr, te in skf.split(np.zeros(len(y)), y):
        Xtr, Xte = build_features_fn(tr, te)
        if c_grid is None:
            best_C = 1.0
        else:
            inner_skf = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=seed)
            best_C, best_auc = 1.0, -np.inf
            for C in c_grid:
                inner_scores = np.zeros(len(tr), dtype=float)
                for itr, ite in inner_skf.split(Xtr, y[tr]):
                    sc_in = StandardScaler().fit(Xtr[itr])
                    clf_in = LogisticRegression(C=C, max_iter=2000, random_state=seed,
                                                 class_weight="balanced")
                    clf_in.fit(sc_in.transform(Xtr[itr]), y[tr][itr])
                    inner_scores[ite] = clf_in.predict_proba(sc_in.transform(Xtr[ite]))[:, 1]
                auc_in = roc_auc_score(y[tr], inner_scores)
                if auc_in > best_auc:
                    best_auc, best_C = auc_in, C
        chosen_Cs.append(best_C)
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(C=best_C, max_iter=2000, random_state=seed, class_weight="balanced")
        clf.fit(sc.transform(Xtr), y[tr])
        y_score[te] = clf.predict_proba(sc.transform(Xte))[:, 1]
    return y_score, chosen_Cs

def summarize(name, y_true, y_score):
    auc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    m, lo, hi, pv = bootstrap_auc_ci(y_true, y_score)
    print(f"{name:42s} AUC={auc:.3f} AUPRC={auprc:.3f}  95% CI=[{lo:.3f},{hi:.3f}]  p={pv:.4f}")
    return {"variant": name, "AUC": auc, "AUPRC": auprc, "CI_low": lo, "CI_high": hi, "p_vs_chance": pv}


In [ ]:
clock_X = df_m[CLOCK_GENE_COLS].values

def make_feature_builder(compositor_fit_fn, use_sign_ref=False):
    def build(tr, te):
        if use_sign_ref:
            f = compositor_fit_fn(clock_X[tr], sign_ref=y[tr])
        else:
            f = compositor_fit_fn(clock_X[tr])
        return f(clock_X[tr]), f(clock_X[te])
    return build

results_composite = []
for name, fn, use_sign in [
    ("Circadian_zscore", zscore_composite_fit, False),
    ("Circadian_rank", rank_composite_fit, False),
    ("Circadian_PCA1", pca_composite_fit, True),
]:
    sc, _ = nested_cv_pipeline(make_feature_builder(fn, use_sign), y, n_folds=5)
    results_composite.append(summarize(name, y, sc))

composite_compare = pd.DataFrame(results_composite)
composite_compare


## 4. Signature Stacking: GEP, TIDE-proxy, IPRES, Basal/Luminal Subtype (nested selection + nested C-grid)

In [ ]:
raw_expr_all = pd.read_csv(p("imvigor210_expression.csv"), index_col=0)
log_cpm_all = cpm_log_normalize(raw_expr_all)
common_p = [pt for pt in df_m.index if pt in log_cpm_all.columns]
df_m2 = df_m.loc[common_p]
y2 = df_m2["label"].values.astype(int)

def gene_composite(genes, inverse_genes=None):
    present = [g for g in genes if g in log_cpm_all.index]
    vals = [log_cpm_all.loc[present, common_p].T.values]
    signs = [1.0]
    if inverse_genes:
        inv_present = [g for g in inverse_genes if g in log_cpm_all.index]
        if inv_present:
            vals.append(-log_cpm_all.loc[inv_present, common_p].T.values)
    combined = np.hstack(vals) if len(vals) > 1 else vals[0]
    return composite_score(combined), present

gep_X, gep_present = gene_composite(GEP_GENES)
ipres_X, ipres_present = gene_composite(IPRES_GENES, IPRES_INVERSE_GENES)

ctl_X, _ = gene_composite(TIDE_CTL_GENES)
caf_X, _ = gene_composite(TIDE_CAF_GENES)
mdsc_X, _ = gene_composite(TIDE_MDSC_GENES)
m2_X, _ = gene_composite(TIDE_M2_GENES)
exclusion_X = np.hstack([caf_X, mdsc_X, m2_X]).mean(axis=1, keepdims=True)
tide_proxy_X = exclusion_X - ctl_X

luminal_X, _ = gene_composite(LUMINAL_GENES)
basal_X, _ = gene_composite(BASAL_GENES)
subtype_X = luminal_X - basal_X

tmb_X2 = df_m2[["TMB"]].values.astype(float)
clock_X2 = df_m2[CLOCK_GENE_COLS].values

print(f"GEP genes found: {len(gep_present)}/{len(GEP_GENES)}")
print(f"IPRES genes found: {len(ipres_present)}/{len(IPRES_GENES) + len(IPRES_INVERSE_GENES)}")

feature_blocks = {
    "TMB": tmb_X2,
    "GEP": gep_X,
    "TIDE": tide_proxy_X,
    "IPRES": ipres_X,
    "Subtype": subtype_X,
}

def circadian_block(tr, te):
    gene_scores = [(abs(pointbiserialr(y2[tr], clock_X2[tr, j])[0]), j) for j in range(clock_X2.shape[1])]
    top2 = [j for _, j in sorted(gene_scores, reverse=True)[:2]]
    f = pca_composite_fit(clock_X2[tr][:, top2], sign_ref=y2[tr])
    return f(clock_X2[tr][:, top2]), f(clock_X2[te][:, top2])

COMBOS = {
    "TMB_only": ["TMB"],
    "TMB_plus_GEP": ["TMB", "GEP"],
    "TMB_plus_GEP_plus_TIDE": ["TMB", "GEP", "TIDE"],
    "TMB_plus_GEP_plus_IPRES": ["TMB", "GEP", "IPRES"],
    "TMB_plus_GEP_plus_Subtype": ["TMB", "GEP", "Subtype"],
    "TMB_plus_GEP_plus_AllSignatures": ["TMB", "GEP", "TIDE", "IPRES", "Subtype"],
    "TMB_plus_GEP_plus_NestedCircadianPCA": ["TMB", "GEP", "CIRCADIAN"],
    "TMB_plus_GEP_plus_AllSignatures_plus_CircadianPCA": ["TMB", "GEP", "TIDE", "IPRES", "Subtype", "CIRCADIAN"],
}

def make_combo_builder(block_names):
    def build(tr, te):
        parts_tr, parts_te = [], []
        for b in block_names:
            if b == "CIRCADIAN":
                ctr, cte = circadian_block(tr, te)
            else:
                ctr, cte = feature_blocks[b][tr], feature_blocks[b][te]
            parts_tr.append(ctr); parts_te.append(cte)
        return np.hstack(parts_tr), np.hstack(parts_te)
    return build

rows = []
for name, blocks in COMBOS.items():
    sc, chosen_Cs = nested_cv_pipeline(make_combo_builder(blocks), y2, n_folds=5, c_grid=C_GRID)
    r = summarize(name, y2, sc)
    r["median_selected_C"] = float(np.median(chosen_Cs))
    rows.append(r)

enrich_summary = pd.DataFrame(rows)
enrich_summary


## 5. Independent Replication, Pooled Across Cohorts (GSE78220 + GSE176307) + Power Analysis

In [ ]:
def run_loocv_nested(build_fn, y, seed=RANDOM_SEED):
    loo = LeaveOneOut()
    n = len(y)
    yt, ys = [], []
    for tr, te in loo.split(np.zeros(n)):
        Xtr, Xte = build_fn(tr, te)
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(C=1.0, max_iter=2000, random_state=seed, class_weight="balanced")
        clf.fit(sc.transform(Xtr), y[tr])
        ys.append(clf.predict_proba(sc.transform(Xte))[:, 1][0])
        yt.append(y[te][0])
    return np.array(yt), np.array(ys)

def run_kfold_nested(build_fn, y, n_folds=5, seed=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    y_score = np.zeros(len(y), dtype=float)
    for tr, te in skf.split(np.zeros(len(y)), y):
        Xtr, Xte = build_fn(tr, te)
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(C=1.0, max_iter=2000, random_state=seed, class_weight="balanced")
        clf.fit(sc.transform(Xtr), y[tr])
        y_score[te] = clf.predict_proba(sc.transform(Xte))[:, 1]
    return y_score

cohorts_for_pooling = {"GSE78220": gse78220}
if gse176307 is not None:
    cohorts_for_pooling["GSE176307"] = gse176307
print(f"Cohorts available for circadian-replication pooling: {list(cohorts_for_pooling.keys())}")

pooled_rows = []
for cname, cdf in cohorts_for_pooling.items():
    z = (cdf[CLOCK_GENE_COLS] - cdf[CLOCK_GENE_COLS].mean()) / (cdf[CLOCK_GENE_COLS].std() + 1e-8)
    z = z.copy()
    z["label"] = cdf["label"].values
    z["cohort"] = cname
    pooled_rows.append(z)
pooled = pd.concat(pooled_rows, axis=0)
y_pool = pooled["label"].values.astype(int)
pool_clock_X = pooled[CLOCK_GENE_COLS].values
print(f"Pooled circadian-replication cohort: n={len(y_pool)}, responders={y_pool.sum()} "
      f"({y_pool.mean()*100:.1f}%)")

def pool_feature_builder(compositor_fit_fn, cols, use_sign_ref=False):
    idx = [CLOCK_GENE_COLS.index(c) for c in cols]
    def build(tr, te):
        Xc = pool_clock_X[:, idx]
        if use_sign_ref:
            f = compositor_fit_fn(Xc[tr], sign_ref=y_pool[tr])
        else:
            f = compositor_fit_fn(Xc[tr])
        return f(Xc[tr]), f(Xc[te])
    return build

pooled_results = []
use_kfold = len(y_pool) >= 60
for name, cols, fn, use_sign in [
    ("Full17gene_zscore", CLOCK_GENE_COLS, zscore_composite_fit, False),
    ("Full17gene_rank", CLOCK_GENE_COLS, rank_composite_fit, False),
    ("Full17gene_PCA1", CLOCK_GENE_COLS, pca_composite_fit, True),
    ("Focused_TIMELESS_TIPIN_zscore", ["TIMELESS", "TIPIN"], zscore_composite_fit, False),
]:
    if use_kfold:
        ys = run_kfold_nested(pool_feature_builder(fn, cols, use_sign), y_pool)
        yt = y_pool
    else:
        yt, ys = run_loocv_nested(pool_feature_builder(fn, cols, use_sign), y_pool)
    auc = roc_auc_score(yt, ys)
    m, lo, hi, pv = bootstrap_auc_ci(yt, ys)
    pooled_results.append({"variant": name, "n": len(yt), "AUC": auc, "CI_low": lo, "CI_high": hi,
                            "p_vs_chance": pv})
    print(f"{name:32s} n={len(yt):4d} AUC={auc:.3f}  95% CI=[{lo:.3f},{hi:.3f}]  p(vs chance)={pv:.4f}")

gse_composite_compare = pd.DataFrame(pooled_results)

from scipy.stats import norm

def hanley_mcneil_se(auc, n_pos, n_neg):
    q1 = auc / (2 - auc)
    q2 = 2 * auc ** 2 / (1 + auc)
    var = (auc * (1 - auc) + (n_pos - 1) * (q1 - auc ** 2) + (n_neg - 1) * (q2 - auc ** 2)) / (n_pos * n_neg)
    return np.sqrt(var)

def power_for_auc(n, auc_true, prevalence, alpha=0.05):
    n_pos = max(1, round(n * prevalence))
    n_neg = n - n_pos
    se = hanley_mcneil_se(auc_true, n_pos, n_neg)
    z_crit = norm.ppf(1 - alpha / 2)
    z_effect = (auc_true - 0.5) / se
    return norm.cdf(z_effect - z_crit)

pool_prevalence = y_pool.mean()
for n_hyp in [26, len(y_pool), 150, 250]:
    pw = power_for_auc(n_hyp, auc_true=0.62, prevalence=pool_prevalence)
    print(f"n={n_hyp:4d}  power to detect true AUC=0.62 as significantly >0.5: {pw:.2f}")


## 6. Pre-Registered Held-Out Test (frozen model, single primary result)

In [ ]:
HOLD_OUT_SEED = 20240101
train_idx, test_idx = train_test_split(
    np.arange(len(y2)), test_size=0.30, stratify=y2, random_state=HOLD_OUT_SEED
)

Xtr_tmb, Xte_tmb = tmb_X2[train_idx], tmb_X2[test_idx]
Xtr_gep, Xte_gep = gep_X[train_idx], gep_X[test_idx]
Xtr_final = np.hstack([Xtr_tmb, Xtr_gep])
Xte_final = np.hstack([Xte_tmb, Xte_gep])
ytr_final, yte_final = y2[train_idx], y2[test_idx]

inner_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
best_C, best_auc = 1.0, -np.inf
for C in C_GRID:
    inner_scores = np.zeros(len(ytr_final), dtype=float)
    for itr, ite in inner_skf.split(Xtr_final, ytr_final):
        sc_in = StandardScaler().fit(Xtr_final[itr])
        clf_in = LogisticRegression(C=C, max_iter=2000, random_state=RANDOM_SEED, class_weight="balanced")
        clf_in.fit(sc_in.transform(Xtr_final[itr]), ytr_final[itr])
        inner_scores[ite] = clf_in.predict_proba(sc_in.transform(Xtr_final[ite]))[:, 1]
    auc_in = roc_auc_score(ytr_final, inner_scores)
    if auc_in > best_auc:
        best_auc, best_C = auc_in, C

print(f"Frozen model: TMB + GEP, LogisticRegression(C={best_C}) selected via 5-fold CV on train split only")

final_scaler = StandardScaler().fit(Xtr_final)
final_clf = LogisticRegression(C=best_C, max_iter=2000, random_state=RANDOM_SEED, class_weight="balanced")
final_clf.fit(final_scaler.transform(Xtr_final), ytr_final)
test_scores = final_clf.predict_proba(final_scaler.transform(Xte_final))[:, 1]

primary_auc = roc_auc_score(yte_final, test_scores)
primary_auprc = average_precision_score(yte_final, test_scores)
print(f"PRIMARY RESULT (frozen held-out test, n={len(yte_final)}): "
      f"AUC={primary_auc:.3f}  AUPRC={primary_auprc:.3f}")


## 7. External Cross-Cohort Validation (train=IMvigor210, test=GSE176307)

In [ ]:
external_validation_available = gse176307 is not None

if external_validation_available:
    gep_present_176 = [g for g in GEP_GENES if g in g176_expr.columns]
    gep_X_176 = composite_score(g176_expr.loc[common_176, gep_present_176].values)
    tmb_X_176 = gse176307[["TMB"]].values.astype(float)
    y_176 = gse176307["label"].values.astype(int)
    valid_176 = ~np.isnan(tmb_X_176).ravel()
    gep_X_176, tmb_X_176, y_176 = gep_X_176[valid_176], tmb_X_176[valid_176], y_176[valid_176]

    prod_X = np.hstack([tmb_X2, gep_X])
    prod_scaler = StandardScaler().fit(prod_X)
    prod_clf = LogisticRegression(C=best_C, max_iter=2000, random_state=RANDOM_SEED,
                                   class_weight="balanced")
    prod_clf.fit(prod_scaler.transform(prod_X), y2)

    ext_X = np.hstack([tmb_X_176, gep_X_176])
    ext_scores = prod_clf.predict_proba(prod_scaler.transform(ext_X))[:, 1]

    ext_auc = roc_auc_score(y_176, ext_scores)
    ext_auprc = average_precision_score(y_176, ext_scores)
    m, lo, hi, pv = bootstrap_auc_ci(y_176, ext_scores)
    print(f"External cross-cohort validation (train=IMvigor210 full TMB-available n={len(y2)}, "
          f"test=GSE176307 n={len(y_176)}):")
    print(f"TMB_plus_GEP_external            AUC={ext_auc:.3f} AUPRC={ext_auprc:.3f}  "
          f"95% CI=[{lo:.3f},{hi:.3f}]  p={pv:.4f}")

    ext_summary = pd.DataFrame([{
        "section": "External cross-cohort validation (train=IMvigor210, test=GSE176307)",
        "variant": "TMB_plus_GEP_external", "AUC": ext_auc, "AUPRC": ext_auprc,
        "CI_low": lo, "CI_high": hi, "p_vs_chance": pv, "median_selected_C": best_C,
        "n": len(y_176),
    }])
else:
    print("GSE176307 not available -- external cross-cohort validation skipped.")
    ext_summary = pd.DataFrame(columns=["section", "variant", "AUC", "AUPRC", "CI_low", "CI_high",
                                         "p_vs_chance", "median_selected_C", "n"])


## 8. Final Summary

In [ ]:
final_summary = pd.concat([
    composite_compare.assign(section="Circadian composite method (IMvigor210, nested CV)"),
    enrich_summary.assign(section="Signature stacking (IMvigor210, nested CV + C-grid)"),
    gse_composite_compare.assign(section="Independent replication (pooled melanoma+bladder ICI cohorts)"),
    ext_summary,
], ignore_index=True)

cols_order = ["section", "variant", "n", "AUC", "AUPRC", "CI_low", "CI_high",
              "p_vs_chance", "median_selected_C"]
final_summary = final_summary[[c for c in cols_order if c in final_summary.columns]]
print(f"Pre-registered held-out primary result: AUC={primary_auc:.3f}  AUPRC={primary_auprc:.3f}  "
      f"(n_test={len(yte_final)}, model=TMB+GEP, C={best_C})")
final_summary
